# Phần 1 – Chuẩn bị và thống nhất định dạng
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

Notebook này làm 4 việc: (1) đọc 6 bảng bằng hàm dùng chung, (2) kiểm tra dữ liệu nhận từ khâu làm sạch, (3) xem kỹ các điểm cần lưu ý, (4) kiểm tra các quy ước đã chốt (bảng màu, giao diện, cấu trúc bảng kết quả).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # chạy từ thư mục notebooks/

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src import config as cf
from src.data_loader import doc_tat_ca, doc_tong_hop, doc_gia_ngay, loc, thu_tu_ky
from src.kiem_tra import kiem_tra_du_lieu, tom_tat
from src.ket_qua import CAU_TRUC, khung_rong

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1. Đọc dữ liệu
Đọc 2 lần: bản **không lọc** để kiểm tra đầy đủ, bản **đã lọc 2021–2025** để phân tích.

In [2]:
ds_goc = doc_tat_ca(loc_giai_doan=False)
ds = doc_tat_ca()
pd.DataFrame({ten: {"so_dong_goc": len(ds_goc[ten]), "so_dong_2021_2025": len(ds[ten]),
                    "so_cot": ds[ten].shape[1]} for ten in ds}).T

,so_dong_goc,so_dong_2021_2025,so_cot
tai_chinh_quy,160,160,33
gia_ngay,10851,9976,11
gia_quy,160,160,21
su_kien_chi_tiet,121,121,15
su_kien_quy,160,160,19
tong_hop,160,160,60


In [3]:
b4 = ds["tong_hop"]
b4[["ma_nh", "quy", "nam", "ky", "ky_so", "ngay_ky", "loai_hinh_so_huu", "roe", "car", "car_nguon", "car_cong_bo"]].head(6)

,ma_nh,quy,nam,ky,ky_so,ngay_ky,loai_hinh_so_huu,roe,car,car_nguon,car_cong_bo
0,VCB,Q1,2021,2021Q1,1,2021-03-31,Quốc doanh,27.6485,9.60,dien_tiep_ky_truoc,False
1,VCB,Q2,2021,2021Q2,2,2021-06-30,Quốc doanh,15.4007,9.60,dien_tiep_ky_truoc,False
2,VCB,Q3,2021,2021Q3,3,2021-09-30,Quốc doanh,17.1562,9.60,dien_tiep_ky_truoc,False
3,VCB,Q4,2021,2021Q4,4,2021-12-31,Quốc doanh,23.7260,9.31,cong_bo,True
4,VCB,Q1,2022,2022Q1,5,2022-03-31,Quốc doanh,27.9255,9.31,dien_tiep_ky_truoc,False
5,VCB,Q2,2022,2022Q2,6,2022-06-30,Quốc doanh,19.8009,9.31,dien_tiep_ky_truoc,False


## 2. Kiểm tra dữ liệu nhận từ khâu làm sạch

In [4]:
bao_cao = kiem_tra_du_lieu(ds_goc)
print(tom_tat(bao_cao).to_string())
bao_cao

ket_qua
ĐẠT      29
LỖI       0
LƯU Ý     6


,nhom_kiem_tra,noi_dung,ket_qua,chi_tiet
0,1. Cấu trúc,bang1: đủ 160 dòng,ĐẠT,160 dòng
1,1. Cấu trúc,bang1: không trùng khóa ma_nh+quy+nam,ĐẠT,0 dòng trùng
2,1. Cấu trúc,bang1: đủ 8 ngân hàng × 20 quý,ĐẠT,"ACB=20, BID=20, CTG=20, MBB=20, STB=20, TCB=20, VCB=20, VPB=20"
3,1. Cấu trúc,bang2b: đủ 160 dòng,ĐẠT,160 dòng
4,1. Cấu trúc,bang2b: không trùng khóa ma_nh+quy+nam,ĐẠT,0 dòng trùng
5,1. Cấu trúc,bang2b: đủ 8 ngân hàng × 20 quý,ĐẠT,"ACB=20, BID=20, CTG=20, MBB=20, STB=20, TCB=20, VCB=20, VPB=20"
6,1. Cấu trúc,bang3b: đủ 160 dòng,ĐẠT,160 dòng
7,1. Cấu trúc,bang3b: không trùng khóa ma_nh+quy+nam,ĐẠT,0 dòng trùng
8,1. Cấu trúc,bang3b: đủ 8 ngân hàng × 20 quý,ĐẠT,"ACB=20, BID=20, CTG=20, MBB=20, STB=20, TCB=20, VCB=20, VPB=20"
9,1. Cấu trúc,bang4: đủ 160 dòng,ĐẠT,160 dòng


In [5]:
cf.THU_MUC_KET_QUA.mkdir(parents=True, exist_ok=True)
bao_cao.to_csv(cf.THU_MUC_KET_QUA / "bao_cao_kiem_tra_du_lieu.csv", index=False, encoding="utf-8-sig")
assert (bao_cao["ket_qua"] != "LỖI").all(), "Có mục LỖI – báo Hân sửa trước khi phân tích"
print("Không có lỗi, có thể bắt đầu phân tích.")

Không có lỗi, có thể bắt đầu phân tích.


## 3. Xem kỹ các điểm lưu ý
### 3.1 CAR: 32 giá trị thật

In [6]:
car_cb = b4[b4["car_cong_bo"]].pivot_table(index="ma_nh", columns="nam", values="car", observed=True)
print(b4.loc[b4["car_cong_bo"], "car_nguon"].value_counts().to_string())
car_cb

car_nguon
cong_bo         26
tra_thu_cong     6


nam,2021,2022,2023,2024
ma_nh,,,,
VCB,9.31,9.95,11.39,12.16
BID,8.97,8.92,9.18,9.03
CTG,9.14,8.90,9.30,9.60
TCB,15.00,15.20,14.40,15.30
MBB,10.02,11.53,10.75,11.80
ACB,11.23,12.80,12.50,11.80
STB,9.93,9.49,9.11,10.14
VPB,14.30,14.90,17.10,15.50


32 giá trị CAR thật = 26 số VNDirect công bố + 6 số **tra thủ công** (BID, CTG 2022–2024: VNDirect ghi đúng 8,00% nên nhóm đã tra lại từ nguồn ngân hàng/CTCK, link trong `data/manual/car_tra_thu_cong.csv`). Năm 2025 chưa có số công bố.

### 3.2 NPL của VPB và quý lỗ của STB

In [7]:
npl_nam = b4.pivot_table(index="ma_nh", columns="nam", values="npl", aggfunc="mean", observed=True).round(2)
npl_nam

nam,2021,2022,2023,2024,2025
ma_nh,,,,,
VCB,0.86,0.73,0.97,1.15,0.91
BID,1.50,1.13,1.50,1.54,1.81
CTG,1.29,1.31,1.26,1.40,1.26
TCB,0.49,0.66,1.11,1.19,1.17
MBB,0.98,1.08,1.67,1.99,1.65
ACB,0.80,0.83,1.11,1.48,1.20
STB,1.53,1.11,1.86,2.40,3.53
VPB,3.88,5.21,5.88,4.72,3.89


In [8]:
b4.loc[b4["loi_nhuan_sau_thue"] < 0, ["ma_nh", "ky", "loi_nhuan_sau_thue", "roa", "roe", "npl"]]

,ma_nh,ky,loi_nhuan_sau_thue,roa,roe,npl
139,STB,2025Q4,-2752.462,-1.2468,-17.9648,6.4076


### 3.3 Cân bằng quan sát cho kiểm định câu 2

In [9]:
nh_nam = b4.groupby(["loai_hinh_so_huu", "ma_nh", "nam"], observed=True).size().reset_index()
print("Quan sát ngân hàng–năm cho NIM, NPL:", nh_nam.groupby("loai_hinh_so_huu", observed=True).size().to_dict())
print("Giá trị CAR công bố:", b4[b4["car_cong_bo"]].groupby("loai_hinh_so_huu", observed=True).size().to_dict())

Quan sát ngân hàng–năm cho NIM, NPL: {'Quốc doanh': 15, 'Tư nhân': 25}
Giá trị CAR công bố: {'Quốc doanh': 12, 'Tư nhân': 20}


## 4. Kiểm tra quy ước đã chốt
### 4.1 Bảng màu

In [10]:
fig = go.Figure()
for i, nh in enumerate(cf.DS_NGAN_HANG):
    fig.add_trace(go.Bar(x=[nh], y=[1], marker_color=cf.MAU_NGAN_HANG[nh], name=nh,
                         text=f"{cf.TEN_DAY_DU[nh]}<br>{cf.MAU_NGAN_HANG[nh]}", textposition="inside"))
for nhom, mau in cf.MAU_NHOM.items():
    fig.add_trace(go.Bar(x=[nhom], y=[1], marker_color=mau, name=nhom, text=mau, textposition="inside"))
fig.update_yaxes(visible=False)
cf.ap_dung_giao_dien(fig, "Bảng màu cố định: 8 ngân hàng và 2 nhóm sở hữu", chu_thich="")
fig.update_layout(showlegend=False, height=300)
fig.show()

### 4.2 Biểu đồ mẫu theo đúng hợp đồng hàm vẽ
Hàm mẫu dưới đây theo đúng 6 quy tắc trong `QUY_UOC.md`; `viz.py` ở Phần 2 sẽ viết theo mẫu này.

In [11]:
def ve_duong_chi_so(df, chi_so="roe", ma_nh=None, nam=None):
    d = loc(df, ma_nh=ma_nh, nam=nam).copy()
    d["ma_nh"] = d["ma_nh"].astype(str)
    fig = px.line(d, x="ky", y=chi_so, color="ma_nh", markers=True,
                  color_discrete_map=cf.MAU_NGAN_HANG,
                  category_orders={"ma_nh": cf.DS_NGAN_HANG, "ky": thu_tu_ky()})
    return cf.ap_dung_giao_dien(fig, f"{cf.NHAN_CHI_SO[chi_so]} theo quý", "Quý", cf.NHAN_CHI_SO[chi_so])

ve_duong_chi_so(b4, "nim", ma_nh=["VCB", "BID", "CTG", "TCB"], nam=[2023, 2024, 2025]).show()

### 4.3 Giá ngày: mốc chuẩn hóa

In [12]:
gia = ds["gia_ngay"]
print("Giai đoạn:", gia["ngay"].min().date(), "→", gia["ngay"].max().date())
print("Phiên gốc (=100) của từng mã:", gia.groupby("ma_nh", observed=True)["ngay"].min().dt.date.unique())

Giai đoạn: 2021-01-04 → 2025-12-31
Phiên gốc (=100) của từng mã: [datetime.date(2021, 1, 4)]


### 4.4 Khung 4 bảng kết quả

In [13]:
for ten, cot in CAU_TRUC.items():
    print(f"{ten}.csv ({len(cot)} cột): {', '.join(cot)}\n")

thong_ke_mo_ta.csv (13 cột): chi_so, cap_do, ma_nh, nam, loai_hinh_so_huu, mau_du_lieu, so_quan_sat, trung_binh, trung_vi, do_lech_chuan, nho_nhat, lon_nhat, don_vi

xep_hang_nam.csv (8 cột): chi_so, nam, ma_nh, loai_hinh_so_huu, gia_tri, hang, hang_nam_truoc, thay_doi_hang

kiem_dinh_nhom.csv (24 cột): chi_so, mau_du_lieu, don_vi_quan_sat, phan_tich, n_quoc_doanh, n_tu_nhan, tb_quoc_doanh, tb_tu_nhan, trung_vi_quoc_doanh, trung_vi_tu_nhan, shapiro_p_quoc_doanh, shapiro_p_tu_nhan, levene_p, kiem_dinh, thong_ke, p_value, muc_y_nghia, bac_bo_H0, gia_thuyet_H0, gia_thuyet_H1, ket_luan, chenh_lech_tb, ten_hieu_ung, do_lon_hieu_ung

tuong_quan.csv (13 cột): chi_so, bien_so_sanh, mau_du_lieu, phan_tich, so_quan_sat, pearson_r, pearson_p, pearson_p_hieu_chinh, spearman_rho, spearman_p, spearman_p_hieu_chinh, phuong_phap_hieu_chinh, xep_hang_do_manh



## Kết luận Phần 1
- Dữ liệu đạt toàn bộ kiểm tra bắt buộc (160 dòng, không trùng khóa, cột chính đủ, 32 CAR công bố, các bảng nhất quán).
- CAR: 6 giá trị của BID, CTG 2022–2024 là số tra thủ công (thay số 8,00% của VNDirect).
- Ghi nhớ khi phân tích: STB lỗ Q4/2025; VPB có NPL cao cấu trúc; CAR 2025 không phải số công bố.
- Quy ước màu, hàm đọc, hợp đồng hàm vẽ và cấu trúc 4 bảng kết quả đã chốt trong `src/` và `QUY_UOC.md`.